In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm import tqdm

from collections import Counter
from collections import defaultdict

from albumentations.pytorch import ToTensorV2
import albumentations as A

import cv2
import numpy as np
import timm

import random
import os
from glob import glob

d:\LabsMIET\MLlab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 9999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
class_to_idx = { "Апельсин": 0,
                 "Бананы": 1,
                 "Груши": 2, 
                 "Кабачки": 3, 
                 "Капуста": 4, 
                 "Картофель": 5, 
                 "Киви": 6, 
                 "Лимон": 7, 
                 "Лук": 8, 
                 "Мандарины": 9, 
                 "Морковь": 10, 
                 "Огурцы": 11, 
                 "Томаты": 12, 
                 "Яблоки зелёные": 13, 
                 "Яблоки красные": 14 }

In [4]:
class MyDataset(Dataset):
    def __init__(self, images_filepaths, name2label, transform=None):
        self.images_filepaths = images_filepaths
        self.transform = transform
        self.name2label = name2label

    def __len__(self):
        return len(self.images_filepaths)

    def __getitem__(self, idx):
        image_filepath = self.images_filepaths[idx]
        image = cv2.imdecode(np.fromfile(image_filepath, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.name2label[os.path.normpath(image_filepath).split(os.sep)[-3]]
        
        if self.transform is not None:
            image = self.transform(image=image)['image']
        return image, label


def train_test_split_from_directory(root_path, folder2class, train_size=0.8):
    train, test = [], []

    for class_name in os.listdir(root_path):
        class_path = os.path.join(root_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for subclass_name in os.listdir(class_path):
            subclass_path = os.path.join(class_path, subclass_name)
            if not os.path.isdir(subclass_path):
                continue

            images = glob(os.path.join(subclass_path, '*.jpg')) + \
                     glob(os.path.join(subclass_path, '*.png')) + \
                     glob(os.path.join(subclass_path, '*.jpeg'))
            
            if len(images) == 0:
                continue
            
            # делим подклассы в пропорции 80/20
            random.shuffle(images)
            split_idx = int(train_size * len(images))

            if split_idx == 0 and len(images) > 0:
                split_idx = 1

            train.extend(images[:split_idx])
            test.extend(images[split_idx:])

    random.shuffle(train)
    random.shuffle(test)

    return train, test

Настройка датасета, writer для вывода, device - на чем обучается

In [5]:
dataset_path = 'train/train'
train, test = train_test_split_from_directory(dataset_path, class_to_idx)

writer = SummaryWriter("kirillLogs")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Функция ошибки

In [6]:
class FocalSmoothingLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits, targets):
        n_classes = logits.size(-1)
        
        # Label smoothing  
        with torch.no_grad():
            true_dist = torch.full_like(logits, self.label_smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        
        # Log softmax 
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Cross entropy с label smoothing
        ce_loss = -(true_dist * log_probs).sum(dim=-1)        
        
        # Focal-часть 
        pt = torch.exp(-ce_loss)                                 
        modulating_factor = (1 - pt) ** self.gamma
        
        #  Class weights 
        if self.alpha is not None:
            if self.alpha.dim() == 1:  
                alpha_t = self.alpha[targets]
            else:
                alpha_t = self.alpha
            loss = alpha_t * modulating_factor * ce_loss
        else:
            loss = modulating_factor * ce_loss
        
        # Редукция
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

Модель

In [7]:
class HierarchicalSwinV2(nn.Module):
    def __init__(self, num_classes=15, model_name="swinv2_cr_small_ns_224.sw_in1k"):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True  
        )

        feature_channels = self.backbone.feature_info.channels()

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(feature_channels[0], 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(0.2),
            ),
            nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(feature_channels[1], 160),
                nn.LayerNorm(160),
                nn.GELU(),
                nn.Dropout(0.2),
            ),
            nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(feature_channels[2], 224),
                nn.LayerNorm(224),
                nn.GELU(),
                nn.Dropout(0.3),
            ),
            nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(feature_channels[3], 320),
                nn.LayerNorm(320),
                nn.GELU(),
                nn.Dropout(0.3),
            ),
        ])
        
        self.classifier = nn.Sequential(
            nn.Linear(832, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

        total_dim = 832  

        self.fusion_attention = nn.Sequential(
            nn.Linear(total_dim, total_dim // 4),
            nn.GELU(),
            nn.Linear(total_dim // 4, total_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.backbone(x)

        pooled = []
        for i in range(4):
            feat = features[i]
            out = self.heads[i](feat)
            pooled.append(out)

        fused = torch.cat(pooled, dim=1)

        attention = self.fusion_attention(fused)

        fused = fused * (1 + attention)

        logits = self.classifier(fused)
        return logits

Классы, классы весов, sampling

Аугментация

In [8]:
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.75, 1.0)),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),

    A.Affine(
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
        scale=(0.9, 1.1),      
        rotate=(-100, 100),          
        border_mode=0,
        value=0,
        p=0.85
    ),

    A.ColorJitter( # Гамма
        brightness=0.25,
        contrast=0.25, 
        saturation=0.25, 
        hue=0.01, 
        p=0.5
    ),

    A.RandomShadow(p=0.25), # Тени
    A.RandomFog(p=0.15, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман

    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), sigma_limit=(0.1, 1.5)),
        A.GaussNoise(var_limit=(5.0, 30.0)),
    ], p=0.8),

    A.CoarseDropout(
        num_holes_range=(1, 6),
        hole_height_range=(0.03, 0.12),
        hole_width_range=(0.03, 0.12),
        fill=128,
        p=0.7
    ),

    A.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

C:\Users\Kirill\AppData\Local\Temp\ipykernel_5924\3045451763.py:7: UserWarning: Argument(s) 'value' are not valid for transform Affine
  A.Affine(
C:\Users\Kirill\AppData\Local\Temp\ipykernel_5924\3045451763.py:25: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(p=0.15, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман
C:\Users\Kirill\AppData\Local\Temp\ipykernel_5924\3045451763.py:29: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0)),


Датасеты, лоадеры

In [9]:
train_dataset = MyDataset(
    images_filepaths=train, 
    name2label=class_to_idx, 
    transform=train_transforms
)
test_dataset = MyDataset(
    images_filepaths=test, 
    name2label=class_to_idx, 
    transform=val_transforms
)

In [10]:
def count_classes(image_paths, class_to_idx):
    counts = Counter()
    for path in image_paths:
        class_name = os.path.normpath(path).split(os.sep)[-3]
        class_idx = class_to_idx[class_name]
        counts[class_idx] += 1
    return counts

train_counts = count_classes(train, class_to_idx)
test_counts  = count_classes(test, class_to_idx)

total_samples = sum(train_counts.values())         
num_classes = len(train_counts)

class_weights = torch.tensor(
    [total_samples / (num_classes * train_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
)

class_weights = class_weights / class_weights.mean()

# train_labels = torch.tensor([label for _, label in train_dataset])

# weigth_sampling = class_weights[train_labels]

# sampler_weigths = WeightedRandomSampler(
#     weights=weigth_sampling,
#     num_samples=len(weigth_sampling),
#     replacement=True
# )


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    # sampler=sampler_weigths,
    shuffle=True,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False  
)
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False
)

Функция для обучения

Инициализация модели

Обучение

Тест

In [12]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.005, verbose=True, restore_best_weights=True):
        """
        Args:
            patience (int): Количество эпох ожидания без улучшения перед остановкой.
            min_delta (float): Минимальное изменение, считающееся улучшением.
            verbose (bool): Если True, печатает сообщения.
            restore_best_weights (bool): Если True, восстанавливает лучшие веса модели.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.restore_best_weights = restore_best_weights
        
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss  # Для минимизации loss (можно изменить для accuracy)

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.best_score:.6f} --> {-val_loss:.6f}). Saving model ...')
        self.best_model_state = model.state_dict()

In [13]:
@torch.no_grad()
def evaluate(model, dataloader, loss_fn, device, desc="Val"):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc=desc, leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        y_pred = logits.argmax(dim=1)
        total_correct += (y_pred == labels).sum().item()
        total_samples += batch_size

        avg_loss = total_loss / max(total_samples, 1)
        acc = total_correct / max(total_samples, 1)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{acc:.4f}")

    avg_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)
    return accuracy, avg_loss

def training_with_stage(model, criterion, train_loader, val_loader, early_stopping, device, writer, n_epoch=30):
    num_iter = 0
    print('Stage 1', n_epoch)

    head_params = [p for n, p in model.named_parameters() 
                   if "heads" in n or "classifier" in n]

    optimizer_stage1 = optim.AdamW(head_params, lr=1e-3, weight_decay=0.05)

    scheduler_stage1 = optim.lr_scheduler.OneCycleLR(
        optimizer_stage1,
        max_lr=1e-3,
        epochs=5,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=1e4
    )


    for epoch in range(1, 6):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epoch}", leave=True)

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            optimizer_stage1.zero_grad(set_to_none=True)
            loss.backward()
            
            optimizer_stage1.step()
            scheduler_stage1.step()
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == labels).sum().item()

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # tqdm live-metrics
            pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

            # Логирование (по итерациям)
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

        # Валидация (тоже с tqdm)
        val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epoch}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)

        print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")


    print('Stage 2')
    for param in model.backbone.parameters():
        param.requires_grad = True

    optimizer_stage2 = optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if "backbone" in n],
         'lr': 6e-6, 'weight_decay': 0.05},
        {'params': [p for n, p in model.named_parameters() if "heads" in n or "classifier" in n],
         'lr': 7e-5, 'weight_decay': 0.05},
    ])

    scheduler_stage2 = torch.optim.lr_scheduler.OneCycleLR(
        optimizer_stage2,
        max_lr=[6e-6, 7e-5],               
        epochs=n_epoch - 5,         
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )


    for epoch in range(6, n_epoch + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epoch}", leave=True)

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            optimizer_stage2.zero_grad(set_to_none=True)
            loss.backward()
            
            optimizer_stage2.step()
            scheduler_stage2.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == labels).sum().item()

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # tqdm live-metrics
            pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

            # Логирование (по итерациям)
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

        # Валидация (тоже с tqdm)
        val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epoch}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)

        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            if early_stopping.restore_best_weights and early_stopping.best_model_state is not None:
                model.load_state_dict(early_stopping.best_model_state)
                print("Restored best model weights.")
            break

        print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    return model

In [14]:
n_epochs = 30

model = HierarchicalSwinV2(num_classes=15).to(device)

params_stage1 = []
for name, param in model.named_parameters():
    if "heads" in name or "classifier" in name:
        params_stage1.append({
            'params': param,
            'lr': 1e-3,
            'weight_decay': 0.1
        })
    else:
        param.requires_grad = False

criterion = FocalSmoothingLoss(
    alpha=class_weights.to(device),
    gamma=2.0,
    label_smoothing=0.05
)

erlstoping = EarlyStopping(patience=10, min_delta=0.005, verbose=True, restore_best_weights=True)

In [15]:
model = training_with_stage(
    model,
    criterion,
    train_loader,
    test_loader,
    erlstoping,
    device,
    writer,
    n_epochs
)

Stage 1 30


Epoch 1/30: 100%|██████████| 504/504 [02:03<00:00,  4.08it/s, train_acc=0.4405, train_loss=1.2885]


Epoch 1/30: val_loss=0.6054  val_acc=0.7105


Epoch 2/30: 100%|██████████| 504/504 [02:01<00:00,  4.16it/s, train_acc=0.6479, train_loss=0.7140]


Epoch 2/30: val_loss=0.4623  val_acc=0.7792


...

EarlyStopping counter: 1 out of 10
Epoch 28/30: val_loss=0.1397  val_acc=0.9583


Epoch 29/30: 100%|██████████| 504/504 [04:41<00:00,  1.79it/s, train_acc=0.9510, train_loss=0.1116]


EarlyStopping counter: 2 out of 10
Epoch 29/30: val_loss=0.1396  val_acc=0.9598


Epoch 30/30: 100%|██████████| 504/504 [04:31<00:00,  1.85it/s, train_acc=0.9556, train_loss=0.1083]
                                                                                     

EarlyStopping counter: 3 out of 10
Epoch 30/30: val_loss=0.1395  val_acc=0.9593


In [16]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def sklearn_report(model, dataloader, device, idx2class=None, digits=4):
    model.eval()

    y_true, y_pred = [], []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)

        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_pred.append(preds)
        y_true.append(labels.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    if idx2class is None:
        target_names = None
        labels = None
    else:
        labels = sorted(idx2class.keys())
        target_names = [idx2class[i] for i in labels]

    rep = classification_report(
        y_true, y_pred,
        labels=labels,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )
    print(rep)

    if idx2class is not None:
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print("\nConfusion Matrix:")
        print(cm)

In [17]:
idx2class = {v: k for k, v in class_to_idx.items()}

sklearn_report(model, test_loader, device, idx2class=idx2class, digits=4)

                precision    recall  f1-score   support

      Апельсин     0.9884    0.9661    0.9771       177
        Бананы     0.9636    0.9815    0.9725       162
         Груши     0.9341    0.8763    0.9043        97
       Кабачки     0.9324    0.9718    0.9517        71
       Капуста     1.0000    0.9762    0.9880       168
     Картофель     0.9277    0.9747    0.9506       158
          Киви     0.9231    0.9351    0.9290        77
         Лимон     0.9627    0.9699    0.9663       133
           Лук     0.9389    0.9318    0.9354       132
     Мандарины     0.9688    0.9936    0.9810       156
       Морковь     0.9667    0.9355    0.9508       124
        Огурцы     0.9741    0.9658    0.9700       117
        Томаты     0.9545    0.9800    0.9671       150
Яблоки зелёные     0.9538    0.9706    0.9621       170
Яблоки красные     0.9568    0.9110    0.9333       146

      accuracy                         0.9593      2038
     macro avg     0.9564    0.9560    0.9559 

In [18]:
test_images_dir = "test_images/test_images"
submission_path = "sample_submission.csv"
output_path = "submissionKirill.csv"

In [19]:
import pandas as pd
submission = pd.read_csv(submission_path)

model.eval()
pred_labels = []

with torch.no_grad():
    for image_id in tqdm(submission["image_id"], desc="Predicting"):
        image_path = os.path.join(test_images_dir, image_id)

        image = cv2.imdecode(
            np.fromfile(image_path, dtype=np.uint8),
            cv2.IMREAD_COLOR
        )
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Когда мы хотим сделать предсказание, нам нужно знать, а какие были преобразования при обучении/тестировании
        if val_transforms is not None:
            image = val_transforms(image=image)["image"]

        image = image.unsqueeze(0).to(device)

        logits = model(image)
        pred_idx = logits.argmax(dim=1).item()

        pred_labels.append(pred_idx)


Predicting: 100%|██████████| 2503/2503 [01:01<00:00, 40.72it/s]


In [20]:
submission["label"] = pred_labels
submission.to_csv(output_path, index=False)

submission.head()


,image_id,label
0,fd343552326b42c5a62c192f32549dc7.jpg,2
1,445ca69812cf44f581cc8a89223af277.jpg,7
2,570626ce4d8f41edb8088f49d40a2195.jpg,7
3,02d4acba92f343d798adcb4958fe684b.jpg,11
4,2d4b8e8f38534a39b0d02c440e917b83.jpg,12
